In [ ]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime


r_seed = 42

np.random.seed(r_seed)
tf.random.set_seed(r_seed)

In [ ]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [ ]:
## get dataset

# select only sensor data from smart watches
data = pd.read_csv('normalizado.csv')

In [ ]:
data.shape


In [ ]:
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
#remove the null class
data=data.dropna()

# **Data Exploration**

In [ ]:
## the distribution of output classes
plt.bar(data["atividade"].value_counts().index, data["atividade"].value_counts().values)
plt.show()

In [ ]:
## x,y,z vaues for activity 'sit' 
data[(data['atividade'] == 0 )&(data['pessoa_x']==1600)][['acelerômetro_x','acelerômetro_Y','acelerômetro_Z']][:200].plot(title='0',subplots=True, figsize=(16, 12))

In [ ]:
## x,y,z vaues for activity 'stand' 
data[(data['atividade'] == 0 )&(data['pessoa_x']==1600)][['giroscópio_x','giroscópio_Y','giroscópio_Z']][:200].plot(title='0',subplots=True, figsize=(16, 12))

In [ ]:
## number of events per user

plt.bar(data["pessoa_x"].value_counts().index, data["pessoa_x"].value_counts().values)
plt.show()

In [ ]:
data = data[data["atividade"].isin([0,1,2,3])]

In [ ]:
data.shape

In [ ]:
data.head()

# **Models**

In [ ]:
## test-train split by keeping 80% for training and 20% for testing
#X = data['acelerômetro_x','acelerômetro_Y','acelerômetro_Z','giroscópio_x','giroscópio_Y','giroscópio_Z']
X = data[['acelerômetro_x','acelerômetro_Y','acelerômetro_Z','giroscópio_x','giroscópio_Y','giroscópio_Z']]
y = data[['atividade']]





In [ ]:
X.head()

In [ ]:
## random split of the data for train-test
#df_train,df_test=train_test_split(data, test_size = 0.20, random_state = r_seed)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=r_seed)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)


In [ ]:
def cria_IN_OUT(data, janela, features):
    OUT_indices = np.arange(janela, len(data), janela)
    #print("imprimindo out_indices")
    #print(OUT_indices)
    OUT = data[OUT_indices]
    #print("imprimindo out")
    #print(OUT)
    IN_indices = [range(i-janela, i) for i in OUT_indices] 
    #print("imprimindo IN_INDICES")
    #print(IN_indices)  
    #print("imprimindo IN_indices flattened") 
    #print(np.array(IN_indices).flatten())
    IN = data[np.array(IN_indices).flatten()]
    #print("imprimindo IN")
    #print(IN)      
    IN = np.reshape(IN, (len(OUT), janela, features))
    IN_final = IN[:, :, :]
    return IN_final

In [ ]:

featuresX = ['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z']
trainX = np.empty((0, 60, len(featuresX)), dtype=float)
testX = np.empty((0, 60, len(featuresX)), dtype=float)
trainY = np.empty((0, 60, 1), dtype=float)
testY = np.empty((0, 60, 1), dtype=float)

array_label_train = y_train.values
print("imprimindo label train length")
print(len(array_label_train))
print("imprimindo label train")
print(array_label_train)
array_label_test = y_test.values
print("imprimindo array label test length")
print(len(array_label_test))
arraytrain = X_train.values
print("imprimindo araytrain")
print(len(arraytrain))
arrayteste = X_test.values
print("imprimindo arraytest")
print(len(arrayteste))
transform = cria_IN_OUT(arraytrain, 60, 6)
transform2 = cria_IN_OUT(arrayteste, 60, 6)
transform3 = cria_IN_OUT(array_label_train, 60, 1)
transform4 = cria_IN_OUT(array_label_test, 60, 1)
trainX = np.concatenate([trainX, transform])
testX = np.concatenate([testX, transform2])
trainY = np.concatenate([trainY, transform3])
testY = np.concatenate([testY, transform4])
print(trainX.shape)
print(trainY.shape)
print(testX.shape)
print(testY.shape)


In [ ]:
''' df_train=df_train[['acelerômetro_x','acelerômetro_Y','acelerômetro_Z','giroscópio_x','giroscópio_Y','giroscópio_Z','atividade']]
df_test=df_test[['acelerômetro_x','acelerômetro_Y','acelerômetro_Z','giroscópio_x','giroscópio_Y','giroscópio_Z','atividade']]

 '''

In [ ]:
''' ## scale x,y,z columns 
feat_cols = ['acelerômetro_x','acelerômetro_Y','acelerômetro_Z','giroscópio_x','giroscópio_Y','giroscópio_Z']
#scale = RobustScaler().fit(df_train[feat_cols])
#df_train.loc[:, feat_cols] = scale.transform(df_train[feat_cols].to_numpy())
#df_test.loc[:, feat_cols] = scale.transform(df_test[feat_cols].to_numpy())
df_train.loc[:, feat_cols] = df_train[feat_cols].to_numpy()
df_test.loc[:, feat_cols] = df_test[feat_cols].to_numpy() '''

In [ ]:
#df_train.head()

In [ ]:
## create input time sequences and take labels of frequently occuring class in the sequence
## hot encode the target variable

''' def input_seq(X, Y, timesteps):
    Xin = np.empty((60, 6), dtype=float)
    Yin = np.empty((60), dtype=float)
    for i in range(0, len(X) - timesteps):
        v = X.iloc[i:(i + timesteps)].values
        labels = Y.iloc[i:(i + timesteps)].values
        Xin = np.concatenate([Xin,v])        
        Yin = np.concatenate([Yin,labels])
    return np.array(Xin), np.array(Yin)


X_tr, Y_tr = input_seq(df_train[['acelerômetro_x','acelerômetro_Y','acelerômetro_Z','giroscópio_x','giroscópio_Y','giroscópio_Z']], df_train['atividade'], 60)
X_te, Y_te = input_seq(df_test[['acelerômetro_x','acelerômetro_Y','acelerômetro_Z','giroscópio_x','giroscópio_Y','giroscópio_Z']], df_test['atividade'], 60)

#hotenc=OneHotEncoder(handle_unknown='ignore', sparse=False) '''
#hotenc= hotenc.fit(Y_tr)
#Y_tr = hotenc.transform(Y_tr)
#Y_te = hotenc.transform(Y_te)





In [ ]:
#X_tr.shape

In [ ]:
#print(X_tr)

In [ ]:
#X_tr = np.array(X_tr).flatten()

In [ ]:
#X_tr.shape


In [ ]:
#Y_tr.shape

In [ ]:
#Y_tr= np.reshape(Y_tr,(603762,60,1))

In [ ]:
#print(Y_tr)


In [ ]:
#Y_tr = np.array(Y_tr).flatten()

In [ ]:
#Y_tr.shape

In [ ]:

''' ## LSTM
model = keras.Sequential()
model.add(keras.layers.LSTM(units=32,input_shape=[X_tr.shape[1], X_tr.shape[2]]))
model.add(keras.layers.Dropout(rate=0.5))
model.add(keras.layers.Dense(Y_tr.shape[1], activation='softmax'))
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['acc'])
#print(model.summary())
history=model.fit(X_tr, Y_tr, epochs=20, batch_size=128,validation_split=0.2,shuffle=False) '''

In [ ]:
optimizer = optimizers.Adam(clipvalue=1.0)

In [ ]:
def cria_LSTM(hidden_units, dense_units, input_shape, activation, optimizer):
  model = tf.keras.Sequential()
  model.add(tf.keras.layers.LSTM(hidden_units, input_shape=input_shape,activation=activation[0], return_sequences=True))
  #model.add(tf.keras.layers.Dense(units=64, activation=activation[0]))
  model.add(tf.keras.layers.Dropout(rate=0.5))
  #model.add(tf.keras.layers.Dense(units=32, activation=activation[0]))
  model.add(tf.keras.layers.Dense(units=dense_units, activation=activation[1]))
  model.compile(loss='sparse_categorical_crossentropy', optimizer ='adam', metrics=['accuracy'])
  return model

def reset_model():
  global model
  model = cria_LSTM(hidden_units=120, dense_units=4, input_shape=(60, 6), activation=['tanh', 'softmax'], optimizer=tf.keras.optimizers.Adam())

model = cria_LSTM(hidden_units=120, dense_units=4, input_shape=(60, 6), activation=['tanh', 'softmax'], optimizer=optimizer)

In [ ]:
historico = []
for i in range(10):
    reset_model()
    print('Rodada %d' % i)
    history = model.fit(trainX, trainY, epochs = 50, batch_size = 50, verbose = 1,  validation_data=(testX, testY))
    historico.append(history)


In [ ]:
plt.figure(figsize= (7,7))

plt.subplot(2,1,1)
plt.title("Loss - {}".format(data))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'], 'k')
plt.legend(['treino', 'teste'])

plt.subplot(2,1,2)
plt.title("acurácia")
plt.ylim(0,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'], 'k')
plt.legend(['treino', 'teste'])
plt.xlabel('epoch')

In [ ]:
nome = 'Acurácia_{}'.format(data)

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(20, 7))
axes = axes.flatten()

for i in range(10):
    ax = axes[i]
    ax.set_title(f'Acurácia - Treino {i + 1}')
    ax.plot(historico[i].history['accuracy'], label='Treino')
    ax.plot(historico[i].history['val_accuracy'], label='Validação')
    ax.set_xlabel('Épocas')
    ax.set_ylabel('Acurácia')
    ax.set_ylim(0, 1)
    ax.legend()

plt.suptitle('Acurácia {}'.format(data))
plt.savefig('Plots/%s' % arq)
plt.tight_layout()

In [ ]:
## test set accuracy
model.evaluate(testX, testY)